# "But wait... there's more"

## A More Visible Agent Loop

The Digital Twin contained an Agent Loop. But it was behind-the-scenes, running every time the user asked a message. Using its tools and then replying. It didn't feel very... loopy.

### Adding 2 more ingredients to make it more real

Let's make an Agent Loop with some familiar features borrowed from Claude Code:

1. A Terminal UI (TUI)
2. A Checklist tool to cause and track multiple tool calls


In [1]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [3]:
openai = OpenAI()

In [4]:
# Some lists!

checklist = []
completed = []

In [5]:
def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result

In [6]:
get_checklist_report()

''

In [7]:
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [8]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    Console().print(completion_notes)
    return get_checklist_report()

In [9]:
checklist, completed = [], []

create_checklist(["Buy groceries", "Finish week 1", "Eat banana"])

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: Buy groceries\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [10]:
mark_complete(1, "bought")

bought

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: [green][strike]Buy groceries[/strike][/green]\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [12]:
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [13]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [14]:
tools = [{"type": "function", "function": create_checklist_json},
        {"type": "function", "function": mark_complete_json}]

In [15]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [16]:
def loop(messages):
    response = openai.chat.completions.create(model="gpt-5.5", messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model="gpt-5.5", messages=messages, tools=tools)
    show(response.choices[0].message.content)

In [17]:
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
You get a rectangular integer matrix and a positive integer r. A rhombic area of size r centered at a cell contains every cell whose Manhattan distance from the center is less than r. In other words, the center plus all cells within r - 1 steps, counting only up, down, left and right moves. A center is valid only if its whole rhombic area lies inside the matrix. You return the largest sum of values over all valid centers.
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [18]:
checklist, completed = [], []
loop(messages)

Checklist #1: Restate the task and identify the geometric shape to sum.
Checklist #2: Derive an efficient way to compute each rhombic/Manhattan-ball sum.
Checklist #3: Handle valid center bounds and edge cases.
Checklist #4: Provide the final algorithm, complexity, and reference implementation.

Done. The area is a Manhattan ball/diamond of radius R = r - 1 centered at a matrix cell.

Checklist #1: Restate the task and identify the geometric shape to sum.
Checklist #2: Derive an efficient way to compute each rhombic/Manhattan-ball sum.
Checklist #3: Handle valid center bounds and edge cases.
Checklist #4: Provide the final algorithm, complexity, and reference implementation.

Done. Use the rotated-coordinate transform u=i+j, v=i-j+(m-1). A Manhattan diamond becomes an axis-aligned square, 
so each sum is one 2D-prefix rectangle query.

Checklist #1: Restate the task and identify the geometric shape to sum.
Checklist #2: Derive an efficient way to compute each rhombic/Manhattan-ball sum.
Checklist #3: Handle valid center bounds and edge cases.
Checklist #4: Provide the final algorithm, complexity, and reference implementation.

Done. Valid centers satisfy R <= i < n-R and R <= j < m-R. If this range is empty, there is no valid center.

Checklist #1: Restate the task and identify the geometric shape to sum.
Checklist #2: Derive an efficient way to compute each rhombic/Manhattan-ball sum.
Checklist #3: Handle valid center bounds and edge cases.
Checklist #4: Provide the final algorithm, complexity, and reference implementation.

Done. Final response includes algorithm, correctness argument, complexity, and Python implementation.

Checklist #1: Restate the task and identify the geometric shape to sum.
Checklist #2: Derive an efficient way to compute each rhombic/Manhattan-ball sum.
Checklist #3: Handle valid center bounds and edge cases.
Checklist #4: Provide the final algorithm, complexity, and reference implementation.

Solution idea

A rhombic area of size r is all cells within Manhattan distance less than r.  
Let:

R = r - 1

So for a center (i, j), we need the sum of all cells (x, y) such that:

|x - i| + |y - j| <= R

A center is valid only if the whole diamond fits inside the matrix.

That means:

R <= i < n - R  
R <= j < m - R

where n is the number of rows and m is the number of columns.

---

Key trick: rotate coordinates

Use transformed coordinates:

u = i + j  
v = i - j + (m - 1)

The offset m - 1 keeps v non-negative.

Under this transformation, the Manhattan diamond:

|x - i| + |y - j| <= R

becomes an axis-aligned square:

u_center - R <= u <= u_center + R  
v_center - R <= v <= v_center + R

So we can place each matrix value into this rotated grid and build a 2D prefix sum over it. Then every rhombus sum 
becomes one rectangle-sum query.

---

Python implementation

def largest_rhombic_sum(matrix, r):
    n = len(matrix)
    m = len(matrix[0])

    R = r - 1

    # No valid center exists
    if 2 * R + 1 > n or 2 * R + 1 > m:
        return None

    # Rotated grid dimensions
    U = n + m - 1
    V = n + m - 1

    rot = [[0] * V for _ in range(U)]

    # Put each matrix value into rotated coordinates
    for i in range(n):
        for j in range(m):
            u = i + j
            v = i - j + (m - 1)
            rot = matrix

    # Build 2D prefix sum
    pref = [[0] * (V + 1) for _ in range(U + 1)]

    for i in range(U):
        for j in range(V):
            pref = (
                rot
                + pref
                + pref
                - pref
            )

    def rect_sum(u1, v1, u2, v2):
        # Inclusive rectangle sum
        return (
            pref
            - pref
            - pref
            + pref
        )

    best = None

    # Try every valid center
    for i in range(R, n - R):
        for j in range(R, m - R):
            u = i + j
            v = i - j + (m - 1)

            current = rect_sum(u - R, v - R, u + R, v + R)

            if best is None or current > best:
                best = current

    return best

---

Example

For:

matrix = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
]

r = 2

The only valid center is (1, 1), and its rhombic area is:

  2
4 5 6
  8

Sum:

2 + 4 + 5 + 6 + 8 = 25

So:

largest_rhombic_sum(matrix, 2) == 25

---

Complexity

Let n be the number of rows and m be the number of columns.

The rotated grid has size approximately:

(n + m) × (n + m)

So the complexity is:

Time: O((n + m)² + nm)  
Space: O((n + m)²)

Each rhombic sum query is O(1).

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>